(User_MolecularSystem_LoadingAndInspect)=
# Loading and inspection

This page is a gentle starting point. We will load a small demo system, display it in Jupyter, and get comfortable with where the *molecular system* lives on the Python side.

If you are new to MolSysViewer, here is the mental model we will use throughout the User Guide:

- You interact with a single Python object called `view` (a {class}`molsysviewer.viewer.MolSysView`).
- `view` talks to the browser, where Mol\* renders the 3D scene.
- The data you are visualizing is a MolSysMT molecular system, available as `view.molsys`.

If you only remember one thing from this page, make it this: **`view` is your handle to both the viewer and the data behind it.**

## A gentle “hello, structure”

To keep things reproducible, we will use a built-in demo system. Demos are exposed as a dictionary-like catalog:

- `msv.demo["pentalanine"]` returns a **fresh** view every time you access it (no shared state).
- This is great for tutorials: you never inherit state from earlier cells.


In [1]:
import molsysviewer as msv

list(msv.demo)

['dialanine', '1TCD', '181L', 'pentalanine', 'chicken_villin_HP35']

In [2]:
view = msv.demo["pentalanine"]
view.show()

In [3]:
from molsysviewer.thirds.jupyter import load_html_in_notebook

load_html_in_notebook("../../../_static/views/demo_pentalanine.html")

## What did we just load?

MolSysViewer stores the underlying molecular system in `view.molsys`.

That object is a native MolSysMT container (`molsysmt.MolSys`). MolSysViewer uses MolSysMT to represent and query the system, while Mol\* focuses on rendering the scene in the browser.

You do not need to be a MolSysMT expert to use MolSysViewer. We will introduce the MolSysMT concepts you need as we go. If you want a deeper dive, see [the MolSysMT documentation](https://www.uibcdf.org/molsysmt).

Two gentle guidelines help avoid confusion:

- If you want to **visualize** something, use MolSysViewer methods.
- If you want to **query** the data (atoms, residues, chains, coordinates…), use MolSysMT helpers through `view.molsys`, `view.whole.select(...)`, `view.whole.info(...)`, and `view.whole.get(...)`.

`view.molsys` is a read-only property (you cannot reassign it), but the object itself may be mutable. If you mutate it directly, you can desynchronize what you see from what you think you loaded. When in doubt: modify your data upstream and call `load(...)` again.

## Form-agnostic loading

MolSysViewer delegates loading to MolSysMT. That is why you can often load “different kinds of things” (a file path, a PDB ID, a trajectory object…), and still get a consistent workflow afterwards.

Internally, MolSysViewer converts what you load into a single, stable representation: `molsysmt.MolSys`. That converted object is what you access as `view.molsys`.

You do not need to call MolSysMT conversion functions manually most of the time. But it helps to know that conversion is happening, because it explains why MolSysViewer stays predictable even when inputs come from different sources.


In [4]:
type(view.molsys)

molsysmt.native.molsys.MolSys

## A quick inspection checklist

When you load a new system, it helps to ask three simple questions:

1. **What is the system, globally?**
2. **Which entity types are present?**
3. **How is it organized into chains?**

MolSysViewer offers two small inspection helpers:

- `view.whole.info(...)` shows a readable summary table in Jupyter.
- `view.whole.get(...)` returns values you can use in Python (arrays, lists, labels…).

Both delegate to MolSysMT and can work at different hierarchical *element* levels (for example: `"atom"`, `"group"`, `"chain"`, `"entity"`).

Let’s answer the first three questions with `view.whole.info(...)`:


In [5]:
view.whole.info(element="system")

ViewerInfo(molsys_section=<pandas.io.formats.style.Styler object at 0x721485d616a0>, view_section=<pandas.io.formats.style.Styler object at 0x721485d2e0d0>)

In [6]:
view.whole.info(element="entity")

ViewerInfo(molsys_section=<pandas.io.formats.style.Styler object at 0x72148768ec40>, view_section=<pandas.io.formats.style.Styler object at 0x72148768eb10>)

And here is a small `get` example: we retrieve the group names for the first chain (`chain_index==0`).

In [7]:
view.whole.get(element="chain", selection="chain_index==0", group_name=True)

[]

For more examples of `info`, `get`, and how they interact with selections, see {doc}`info` and {doc}`get`.

## Loading your own molecular system

When you move from demos to your own work, you will typically load a molecular system from a local file or from an object you already have in memory (for example, an `mdtraj.Trajectory`, an OpenMM topology + coordinates, or a MolSysMT object).

There are two common patterns:

1. Create a new view from a molecular-system-like input with {func}`molsysviewer.new_view`.
2. Create a view first, then call `view.load(...)`.

Both are valid; pick the one that matches your style.

Why this matters: `new_view` can either subset the system or keep it intact. Use `load_mode="selection"` (default) to load only the selection. Use `load_mode="all"` to load everything and create a region tagged `selection`.


In [8]:
view = msv.MolSysView()
view.load('181L')
view.whole.info()

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1
section,tag,kind,visible,active,layer tag,representation,preset,n atoms,n members,n picks,details
whole,whole,whole,True,True,None,None,None,nan,nan,None,
loads,load0,load,None,True,None,None,None,1441.000000,nan,None,atoms 0–1441
styles,current,style,None,False,None,None,None,nan,0.000000,None,builtins=11
active_selection,active_selection,empty,None,False,None,None,None,0.000000,nan,None,empty / none


In [9]:
view = msv.new_view('181L')
view.whole.info()

form,n_atoms,n_groups,n_components,n_chains,n_molecules,n_entities,n_waters,n_ions,n_small_molecules,n_proteins,n_structures
molsysmt.MolSys,1441,302,141,6,141,5,136,2,2,1,1
section,tag,kind,visible,active,layer tag,representation,preset,n atoms,n members,n picks,details
whole,whole,whole,True,True,None,None,None,nan,nan,None,
loads,load0,load,None,True,None,None,None,1441.000000,nan,None,atoms 0–1441
styles,current,style,None,False,None,None,None,nan,0.000000,None,builtins=11
active_selection,active_selection,empty,None,False,None,None,None,0.000000,nan,None,empty / none


And if your main goal is simply to display the structure, just call `view.show()`:

In [10]:
view.show()

In [11]:
from molsysviewer.thirds.jupyter import load_html_in_notebook
load_html_in_notebook("../../../_static/views/demo_181L.html")

## Where to go next

Now that you can load and display a system, the next pages will help you build the mental model:

- {doc}`topology`: what the *topology* is (atoms → residues → chains…).
- {doc}`structures`: what a *structure* is (coordinates, box, time; trajectories vs ensembles).
- {doc}`selection`: how you pick subsets of atoms.
- {doc}`info`: how to get a quick overview of a system.
- {doc}`get`: how to extract attribute values into Python.

If you want more datasets to play with, see {doc}`../demo_systems/index`.
